# Linking filtered Isotherms to gathered MOF CIFs

## Merge Strategy
This workflow merges filtered isotherms with known MOFs by name, then runs synonym rescue from UNKNOWN entries.

What this does:
1. Detects required columns from all three inputs (with fallback aliases).
2. Performs a left join from isotherms to KNOWN MOFs so all isotherm rows remain.
3. For still-unmatched isotherm rows, checks `MOFs_from_CCDC_sorted_UNKNOWN` by `nist_name`.
4. Treats UNKNOWN `csd_identifier` as a synonym MOF name and maps it to a KNOWN MOF entry to recover metadata.
5. Appends KNOWN CIF rows that have no corresponding isotherm (sorted alphabetically).
6. Appends UNKNOWN rows that have no corresponding isotherm, even if they map via synonym (for example, `Cu-HKUST-1`).
7. Applies three highlight colors in Excel:
   - light yellow: isotherm without CIF match
   - light blue: KNOWN CIF without isotherm
   - light pink: UNKNOWN entry without isotherm
8. Exports both files in the data folder:
   - filtered_isotherms_merged_with_CCDC.xlsx (highlighted)
   - filtered_isotherms_merged_with_CCDC.csv

In [7]:
# Step 1 - Load input Excel files and standardize their column names
from pathlib import Path
import shutil
import tempfile
import pandas as pd

# Resolve project/data folder from notebook location
project_dir = Path.cwd()
data_dir = project_dir / "data"

def find_excel_file(folder: Path, stem_name: str) -> Path:
    """Find an Excel file by stem name (e.g., 'filtered_isotherms')."""
    candidates = sorted(
        p for p in folder.glob(f"{stem_name}*")
        if p.suffix.lower() in {".xlsx", ".xls", ".xlsm"} and not p.name.startswith("~$")
    )
    if not candidates:
        raise FileNotFoundError(
            f"No Excel file starting with '{stem_name}' was found in {folder}"
        )
    if len(candidates) > 1:
        print(f"Multiple files found for {stem_name}; using: {candidates[0].name}")
    return candidates[0]

def read_excel_with_fallback(path: Path) -> pd.DataFrame:
    """Read Excel robustly; fallback to temp copy if workbook is temporarily locked."""
    try:
        return pd.read_excel(path)
    except PermissionError:
        with tempfile.TemporaryDirectory() as tmpdir:
            tmp_path = Path(tmpdir) / path.name
            shutil.copy2(path, tmp_path)
            return pd.read_excel(tmp_path)

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize column names to make matching robust."""
    out = df.copy()
    out.columns = (
        out.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return out

def apply_aliases(df: pd.DataFrame, alias_map: dict[str, list[str]]) -> pd.DataFrame:
    """Rename first-found aliases to their canonical column names."""
    out = df.copy()
    for canonical, aliases in alias_map.items():
        if canonical in out.columns:
            continue
        for alias in aliases:
            if alias in out.columns:
                out = out.rename(columns={alias: canonical})
                break
    return out

def ensure_columns(df: pd.DataFrame, required: list[str]) -> pd.DataFrame:
    """Ensure required columns exist; create empty columns when absent."""
    out = df.copy()
    for col in required:
        if col not in out.columns:
            out[col] = pd.NA
    return out

filtered_isotherms_path = find_excel_file(data_dir, "filtered_isotherms")
mofs_known_path = find_excel_file(data_dir, "MOFs_from_CCDC_sorted_KNOWN")
mofs_unknown_path = find_excel_file(data_dir, "MOFs_from_CCDC_sorted_UNKNOWN")

df_iso_raw = read_excel_with_fallback(filtered_isotherms_path)
df_mof_known_raw = read_excel_with_fallback(mofs_known_path)
df_mof_unknown_raw = read_excel_with_fallback(mofs_unknown_path)

df_iso = normalize_columns(df_iso_raw)
df_mof_known = normalize_columns(df_mof_known_raw)
df_mof_unknown = normalize_columns(df_mof_unknown_raw)

# Harmonize schema labels so downstream cells remain stable when headers are renamed
df_iso = apply_aliases(df_iso, {
    "name": ["nist_name", "adsorbent_name", "adsorbent"],
    "hashkey": ["hash_key"],
    "filename": ["isotherm_filename", "file_name"],
    "temperature_(k)": ["temperature", "temp"],
    "doi_link": ["doi"],
})

df_mof_known = apply_aliases(df_mof_known, {
    "name": ["nist_name", "adsorbent_name", "adsorbent"],
    "molecular_formula": ["formula"],
    "has_disorder": ["disorder"],
    "csd_identifier": ["csd_id", "identifier", "csd_refcode"],
})

df_mof_unknown = apply_aliases(df_mof_unknown, {
    "nist_name": ["name", "adsorbent_name", "adsorbent"],
    "csd_identifier": ["csd_id", "identifier", "csd_refcode"],
    "hashkey": ["hash_key"],
    "molecular_formula": ["formula"],
    "has_disorder": ["disorder"],
})
df_mof_unknown = ensure_columns(
    df_mof_unknown,
    ["nist_name", "csd_identifier", "hashkey", "doi", "molecular_formula", "has_disorder"],
)

print("Loaded files:")
print(f"- {filtered_isotherms_path.name}: {len(df_iso)} rows")
print(f"- {mofs_known_path.name}: {len(df_mof_known)} rows")
print(f"- {mofs_unknown_path.name}: {len(df_mof_unknown)} rows")
print()
print("Isotherm columns:", list(df_iso.columns))
print("KNOWN MOF columns:", list(df_mof_known.columns))
print("UNKNOWN MOF columns:", list(df_mof_unknown.columns))

Multiple files found for filtered_isotherms; using: filtered_isotherms.xlsx
Loaded files:
- filtered_isotherms.xlsx: 586 rows
- MOFs_from_CCDC_sorted_KNOWN.xlsx: 307 rows
- MOFs_from_CCDC_sorted_UNKNOWN.xlsx: 33 rows

Isotherm columns: ['#', 'name', 'hashkey', 'filename', 'doi_link', 'temperature_(k)']
KNOWN MOF columns: ['name', 'csd_identifier', 'doi', 'molecular_formula', 'has_disorder', 'unnamed:_5']
UNKNOWN MOF columns: ['nist_name', 'csd_identifier', 'hashkey', 'doi', 'unnamed:_4', 'molecular_formula', 'has_disorder']


In [8]:
# Step 2 - Merge by name, resolve synonyms from UNKNOWN, and export
def pick_column(df: pd.DataFrame, candidates: list[str], logical_name: str, table_name: str) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(
        f"Could not find '{logical_name}' in {table_name}. Tried: {candidates}. "
        f"Available columns: {list(df.columns)}"
    )

iso_name_col = pick_column(df_iso, ["name", "nist_name", "adsorbent_name", "adsorbent"], "name", "filtered_isotherms")
iso_hash_col = pick_column(df_iso, ["hashkey", "hash_key"], "hashkey", "filtered_isotherms")
iso_file_col = pick_column(df_iso, ["filename", "isotherm_filename", "file_name"], "filename", "filtered_isotherms")
iso_temp_col = pick_column(df_iso, ["temperature_(k)", "temperature", "temp"], "temperature", "filtered_isotherms")

known_name_col = pick_column(df_mof_known, ["name", "nist_name", "adsorbent_name", "adsorbent"], "name", "MOFs_from_CCDC_sorted_KNOWN")
known_csd_col = pick_column(df_mof_known, ["csd_identifier", "csd_id", "identifier", "csd_refcode"], "csd identifier", "MOFs_from_CCDC_sorted_KNOWN")
known_doi_col = pick_column(df_mof_known, ["doi", "doi_link"], "doi", "MOFs_from_CCDC_sorted_KNOWN")
known_formula_col = pick_column(df_mof_known, ["molecular_formula", "formula"], "molecular formula", "MOFs_from_CCDC_sorted_KNOWN")
known_disorder_col = pick_column(df_mof_known, ["has_disorder", "disorder"], "has_disorder", "MOFs_from_CCDC_sorted_KNOWN")

unknown_nist_col = pick_column(df_mof_unknown, ["nist_name", "name", "adsorbent_name", "adsorbent"], "nist_name", "MOFs_from_CCDC_sorted_UNKNOWN")
unknown_syn_col = pick_column(df_mof_unknown, ["csd_identifier", "csd_id", "identifier"], "csd_identifier synonym", "MOFs_from_CCDC_sorted_UNKNOWN")
unknown_hashkey_col = pick_column(df_mof_unknown, ["hashkey", "hash_key"], "hashkey", "MOFs_from_CCDC_sorted_UNKNOWN")
unknown_doi_col = pick_column(df_mof_unknown, ["doi", "doi_link"], "doi", "MOFs_from_CCDC_sorted_UNKNOWN")
unknown_formula_col = pick_column(df_mof_unknown, ["molecular_formula", "formula"], "molecular formula", "MOFs_from_CCDC_sorted_UNKNOWN")
unknown_disorder_col = pick_column(df_mof_unknown, ["has_disorder", "disorder"], "has_disorder", "MOFs_from_CCDC_sorted_UNKNOWN")

iso_work = df_iso[[iso_name_col, iso_file_col, iso_hash_col, iso_temp_col]].copy()
iso_work.columns = ["name", "isotherm_filename", "hashkey", "temperature"]

known_work = df_mof_known[[known_name_col, known_csd_col, known_doi_col, known_formula_col, known_disorder_col]].copy()
known_work.columns = ["name", "csd_identifier", "doi", "molecular_formula", "has_disorder"]

unknown_work = df_mof_unknown[[unknown_nist_col, unknown_syn_col, unknown_hashkey_col, unknown_doi_col, unknown_formula_col, unknown_disorder_col]].copy()
unknown_work.columns = ["nist_name", "synonym_name", "hashkey", "doi", "molecular_formula", "has_disorder"]

# Normalize names before matching (case-insensitive)
iso_work["name"] = iso_work["name"].astype(str).str.strip()
known_work["name"] = known_work["name"].astype(str).str.strip()
unknown_work["nist_name"] = unknown_work["nist_name"].astype(str).str.strip()
unknown_work["synonym_name"] = unknown_work["synonym_name"].astype(str).str.strip()

iso_work["name_key"] = iso_work["name"].str.casefold()
known_work["name_key"] = known_work["name"].str.casefold()
unknown_work["nist_key"] = unknown_work["nist_name"].str.casefold()
unknown_work["synonym_key"] = unknown_work["synonym_name"].str.casefold()

# Prefer the most complete KNOWN-MOF record when duplicate names exist
known_work = known_work.replace(r"^\s*$", pd.NA, regex=True)
known_work["_completeness"] = known_work[["csd_identifier", "doi", "molecular_formula", "has_disorder"]].notna().sum(axis=1)
known_work = (
    known_work
    .sort_values(["name_key", "_completeness"], ascending=[True, False])
    .drop_duplicates(subset=["name_key"], keep="first")
    .drop(columns=["_completeness"])
)

# Primary merge: keep all isotherms even with no direct KNOWN match
merged = iso_work.merge(
    known_work[["name_key", "csd_identifier", "doi", "molecular_formula", "has_disorder"]],
    on="name_key",
    how="left",
)
direct_match_mask = merged["csd_identifier"].notna()

# Synonym rescue: use UNKNOWN nist_name -> synonym_name, then synonym_name -> KNOWN entry
synonym_map = (
    unknown_work[["nist_key", "synonym_key"]]
    .replace({"nist_key": {"": pd.NA}, "synonym_key": {"": pd.NA}})
    .dropna()
    .drop_duplicates(subset=["nist_key"], keep="first")
)

known_lookup = known_work[["name_key", "csd_identifier", "doi", "molecular_formula", "has_disorder"]].rename(
    columns={"name_key": "synonym_key"}
)

synonym_info = synonym_map.merge(known_lookup, on="synonym_key", how="left")

merged = merged.merge(
    synonym_info.rename(
        columns={
            "csd_identifier": "csd_identifier_syn",
            "doi": "doi_syn",
            "molecular_formula": "molecular_formula_syn",
            "has_disorder": "has_disorder_syn",
        }
    ),
    left_on="name_key",
    right_on="nist_key",
    how="left",
)

for base_col in ["csd_identifier", "doi", "molecular_formula", "has_disorder"]:
    syn_col = f"{base_col}_syn"
    merged[base_col] = merged[base_col].combine_first(merged[syn_col])

# Track which KNOWN names are consumed either directly or via synonym mapping
known_keys_used_direct = set(merged.loc[direct_match_mask, "name_key"].dropna().tolist())
known_keys_used_synonym = set(merged["synonym_key"].dropna().tolist())
known_keys_used = known_keys_used_direct | known_keys_used_synonym

# Base output rows (all isotherms)
final_isotherm_df = merged[[
    "name",
    "csd_identifier",
    "isotherm_filename",
    "hashkey",
    "doi",
    "molecular_formula",
    "temperature",
    "has_disorder",
]].rename(columns={"name": "absorbent_or_nist_name"})

# Add CIF-only rows: KNOWN MOFs that do not have any corresponding isotherm
known_unmatched = known_work[~known_work["name_key"].isin(known_keys_used)].copy()
cif_only_df = pd.DataFrame(
    {
        "absorbent_or_nist_name": known_unmatched["name"],
        "csd_identifier": known_unmatched["csd_identifier"],
        "isotherm_filename": "",
        "hashkey": "",
        "doi": known_unmatched["doi"],
        "molecular_formula": known_unmatched["molecular_formula"],
        "temperature": "",
        "has_disorder": known_unmatched["has_disorder"],
    }
)
cif_only_df = cif_only_df.sort_values(
    by="absorbent_or_nist_name",
    key=lambda s: s.astype(str).str.casefold(),
)

# Add UNKNOWN rows with no isotherm, regardless of whether synonym resolves
iso_name_keys = set(iso_work["name_key"].dropna().tolist())
unknown_eval = unknown_work.copy()
unknown_eval = unknown_eval.merge(
    known_lookup.rename(
        columns={
            "csd_identifier": "resolved_csd_identifier",
            "doi": "resolved_doi",
            "molecular_formula": "resolved_molecular_formula",
            "has_disorder": "resolved_has_disorder",
        }
    ),
    on="synonym_key",
    how="left",
)

unknown_no_isotherm = unknown_eval[~unknown_eval["nist_key"].isin(iso_name_keys)].copy()
unknown_no_isotherm_df = pd.DataFrame(
    {
        "absorbent_or_nist_name": unknown_no_isotherm["nist_name"],
        "csd_identifier": unknown_no_isotherm["resolved_csd_identifier"],
        "isotherm_filename": "",
        "hashkey": unknown_no_isotherm["hashkey"],
        "doi": unknown_no_isotherm["doi"].combine_first(unknown_no_isotherm["resolved_doi"]),
        "molecular_formula": unknown_no_isotherm["molecular_formula"].combine_first(
            unknown_no_isotherm["resolved_molecular_formula"]
        ),
        "temperature": "",
        "has_disorder": unknown_no_isotherm["has_disorder"].combine_first(
            unknown_no_isotherm["resolved_has_disorder"]
        ),
    }
)
unknown_no_isotherm_df = unknown_no_isotherm_df.drop_duplicates()
unknown_no_isotherm_df = unknown_no_isotherm_df.sort_values(
    by="absorbent_or_nist_name",
    key=lambda s: s.astype(str).str.casefold(),
)

# Keep blanks in nullable fields
for col in ["csd_identifier", "doi", "molecular_formula", "has_disorder", "hashkey"]:
    final_isotherm_df[col] = final_isotherm_df[col].fillna("")
    cif_only_df[col] = cif_only_df[col].fillna("")
    unknown_no_isotherm_df[col] = unknown_no_isotherm_df[col].fillna("")

final_isotherm_df["row_origin"] = final_isotherm_df["csd_identifier"].apply(
    lambda x: "isotherm_without_cif" if x == "" else "isotherm_with_cif"
)
cif_only_df["row_origin"] = "cif_without_isotherm"
unknown_no_isotherm_df["row_origin"] = "unknown_without_isotherm"

final_df_with_flags = pd.concat(
    [final_isotherm_df, cif_only_df, unknown_no_isotherm_df],
    ignore_index=True,
)
final_export_df = final_df_with_flags.drop(columns=["row_origin"])

isotherm_match_count = int((final_df_with_flags["row_origin"] == "isotherm_with_cif").sum())
isotherm_unmatched_count = int((final_df_with_flags["row_origin"] == "isotherm_without_cif").sum())
cif_without_isotherm_count = int((final_df_with_flags["row_origin"] == "cif_without_isotherm").sum())
unknown_without_isotherm_count = int((final_df_with_flags["row_origin"] == "unknown_without_isotherm").sum())

output_xlsx = data_dir / "filtered_isotherms_merged_with_CCDC.xlsx"

def highlight_rows(row: pd.Series) -> list[str]:
    row_type = final_df_with_flags.loc[row.name, "row_origin"]
    if row_type == "isotherm_without_cif":
        color = "background-color: #fff3b0"
    elif row_type == "cif_without_isotherm":
        color = "background-color: #cfe8ff"
    elif row_type == "unknown_without_isotherm":
        color = "background-color: #ffd6e7"
    else:
        color = ""
    return [color] * len(row)

# Save highlighted Excel and plain CSV (fallback if the main XLSX is locked)
actual_output_xlsx = output_xlsx
try:
    final_export_df.style.apply(highlight_rows, axis=1).to_excel(output_xlsx, index=False)
except PermissionError:
    actual_output_xlsx = data_dir / "filtered_isotherms_merged_with_CCDC_latest.xlsx"
    final_export_df.style.apply(highlight_rows, axis=1).to_excel(actual_output_xlsx, index=False)

# Keep final_df in kernel as the exported table
final_df = final_export_df

print(f"Total rows in final table: {len(final_df)}")
print(f"Isotherms matched to CIFs: {isotherm_match_count}")
print(f"Isotherms without CIFs (yellow): {isotherm_unmatched_count}")
print(f"CIFs without isotherms (blue, alphabetical): {cif_without_isotherm_count}")
print(f"UNKNOWN without isotherms (pink): {unknown_without_isotherm_count}")
print(f"Saved Excel (highlighted): {actual_output_xlsx}")
final_df.tail(10)

C:\Users\james\AppData\Local\Temp\ipykernel_20380\1847753970.py:163: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  "has_disorder": unknown_no_isotherm["has_disorder"].combine_first(


Total rows in final table: 625
Isotherms matched to CIFs: 397
Isotherms without CIFs (yellow): 189
CIFs without isotherms (blue, alphabetical): 38
UNKNOWN without isotherms (pink): 1
Saved Excel (highlighted): c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\ML Project Directory\Data Collection\data\filtered_isotherms_merged_with_CCDC.xlsx


,absorbent_or_nist_name,csd_identifier,isotherm_filename,hashkey,doi,molecular_formula,temperature,has_disorder
615,Ni3(pzdc)2(7Hade)2(H2O)0.6,UNEZAP,,,10.1039/c0cc05559j,"(C20 H20 N14 Ni3 O12)n,2.18n(H2 O1)",,0.0
616,NJU-Bai10,KIRFOI,,,10.1021/cg400449c,(C63 H39 Cu6 N3 O30)n,,1.0
617,PCN-14,XITYOP,,,"10.1021/Ja4045289', '10.1039/C4ra12432d', '10....",,,
618,ZJU-31a,HUNCIE,,,10.1021/acs.cgd.5b00675,"(C48 H34 Cu2 O12)n,18n(C4 H9 N1 O1),19n(H2 O1)",,1.0
619,ZJU-8a,FUWQIZ,,,10.1039/c5ra12700a,(C22 H15 Cu2 N1 O10)n,,1.0
620,Zn(II)-MOF [Zn(HPylmDC)(DMA)]n,HOFSUS,,,10.1039/c4dt00307a,(C14 H14 N4 O5 Zn1)n,,0.0
621,ZNJU-60a,AHOLAM,,,10.1039/c5ce00930h,(C42 H30 Cu3 N3 O21 P3)n,,1.0
622,{[NH2(CH3)2][Zn(atz)(ox)]$H2O}n,BUBJUF,,,10.1039/c5ra02935j,(C6 H4 N10 O8 Zn2)n,,1.0
623,{[Zn4O(bfbpdc)3-(bpy)0.5(H2O)]*(3DMF)(H2O)}n,WIKGII,,,10.1021/ic302645r,"(C106 H48 F36 N2 O28 Zn8)n,6n(C3 H7 N1 O1),2n(...",,1.0
624,Cu-HKUST-1,FUNGAZ,,NIST-MATDB-3b3e8d3742fee1969162f75b74614acc,10.1021/acs.jpclett.5b00893,,,


## Step 3 - Import manual merge file and isolate valid CSD identifiers

This section prepares the exact list of CSD identifiers that should be used for CIF retrieval.

Workflow used:
1. Automatically locate the newest manual merge file in the `data` folder using the stem `filtered_isotherm_merged_with_CCDC_manual` (and close-name fallback).
2. Load the table from Excel (`.xlsx/.xls/.xlsm`) or CSV.
3. Detect highlighted rows when the source is Excel by reading cell styles with `openpyxl`.
4. Exclude highlighted rows from the retrieval pool (as requested).
5. Extract and normalize `csd_identifier` values (trim, uppercase, drop blanks and duplicates).
6. Save audit outputs in `data`:
   - `filtered_isotherm_merged_with_CCDC_manual_unhighlighted.xlsx`
   - `csd_identifiers_for_mosaec.csv`

Notes:
- If the manual file is CSV, row highlighting is not available in the file format; in that case the code keeps all rows and prints a warning.
- If a `row_origin` column exists, only `isotherm_with_cif` rows are retained as a safety filter.

In [9]:
# Step 3 code - load fixed manual file, ignore highlighted rows, and extract CSD identifiers
from pathlib import Path
import pandas as pd

project_dir = Path.cwd()
data_dir = project_dir / "data"

manual_path = data_dir / "filtered_isotherms_merged_with_CCDC_manual.xlsx"
if not manual_path.exists():
    raise FileNotFoundError(
        f"Required manual merge file was not found: {manual_path}"
    )


def is_highlight_fill(fill) -> bool:
    """Return True if cell fill appears to be manually highlighted."""
    if fill is None:
        return False
    fill_type = getattr(fill, "fill_type", None)
    return fill_type not in (None, "none")


def highlight_mask_from_excel(path: Path, n_rows: int) -> pd.Series:
    """Build a boolean mask: True when a row contains at least one highlighted cell."""
    from openpyxl import load_workbook

    wb = load_workbook(path, data_only=True)
    ws = wb.active

    # DataFrame row 0 corresponds to Excel row 2 (row 1 is header).
    highlighted = []
    for excel_row in range(2, 2 + n_rows):
        row_cells = ws[excel_row]
        highlighted.append(any(is_highlight_fill(cell.fill) for cell in row_cells))

    return pd.Series(highlighted, dtype=bool)


manual_df = pd.read_excel(manual_path)
manual_df.columns = [str(c).strip() for c in manual_df.columns]

if "csd_identifier" not in manual_df.columns:
    raise KeyError(
        f"'csd_identifier' column not found in {manual_path.name}. "
        f"Available columns: {list(manual_df.columns)}"
    )

row_is_highlighted = highlight_mask_from_excel(manual_path, len(manual_df))
unhighlighted_df = manual_df.loc[~row_is_highlighted].copy()

# Safety filter when row_origin exists from prior pipeline steps.
if "row_origin" in unhighlighted_df.columns:
    unhighlighted_df = unhighlighted_df[unhighlighted_df["row_origin"].astype(str).str.strip() == "isotherm_with_cif"].copy()

csd_series = (
    unhighlighted_df["csd_identifier"]
    .astype(str)
    .str.strip()
    .str.upper()
)

valid_csd_ids = sorted(
    {
        cid for cid in csd_series
        if cid and cid.lower() != "nan" and cid.lower() != "none"
    }
)

csd_ids_df = pd.DataFrame({"csd_identifier": valid_csd_ids})

cleaned_table_path = data_dir / "filtered_isotherm_merged_with_CCDC_manual_unhighlighted.xlsx"
csd_id_list_path = data_dir / "csd_identifiers_for_mosaec.csv"

unhighlighted_df.to_excel(cleaned_table_path, index=False)
csd_ids_df.to_csv(csd_id_list_path, index=False)

print(f"Manual source file: {manual_path.name}")
print(f"Total rows in source: {len(manual_df)}")
print(f"Highlighted rows excluded: {int(row_is_highlighted.sum())}")
print(f"Rows retained for CIF retrieval: {len(unhighlighted_df)}")
print(f"Unique valid CSD identifiers: {len(valid_csd_ids)}")
print(f"Saved unhighlighted table: {cleaned_table_path}")
print(f"Saved CSD list: {csd_id_list_path}")

csd_ids_df.head(10)

Manual source file: filtered_isotherms_merged_with_CCDC_manual.xlsx
Total rows in source: 435
Highlighted rows excluded: 22
Rows retained for CIF retrieval: 413
Unique valid CSD identifiers: 274
Saved unhighlighted table: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\ML Project Directory\Data Collection\data\filtered_isotherm_merged_with_CCDC_manual_unhighlighted.xlsx
Saved CSD list: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\ML Project Directory\Data Collection\data\csd_identifiers_for_mosaec.csv


,csd_identifier
0,ACOCOM
1,ADOGEH
2,ADUROI
3,AJORAT
4,ALAMUW
5,AMAFOK
6,AMUSIO01
7,AQIXEF
8,ARAJEK
9,AROFET


## Step 4 - Retrieve CIFs from local MOSAEC database (neutral and charged)

This step replaces the online GitHub retrieval and uses your local MOSAEC database copy.

Local database root used:
`C:/Users/james/OneDrive - Aix-Marseille Université/CNE Wroclaw 2026/CNE Thesis 2026/Scraped DBs/MOSAEC-DB/mosaec-db_base/mosaec-db_base/full`

How this works:
1. Read CSD identifiers prepared in Step 3 (`valid_csd_ids`).
2. Scan both subfolders: `neutral` and `charged`.
3. For each `.cif` filename, use only the first 6 characters of the stem as the CSD identifier key.
4. Match these keys against our target CSD identifier list.
5. Copy matched CIFs into `data/MOSAEC_CIFs_local/`.
6. Create an audit table with a folder-source column showing whether each file came from `neutral` or `charged`.
7. Export not-found CSD identifiers for the next database scan.

Outputs written to `data`:
- `mosaec_local_cif_retrieval_audit.csv`
- `mosaec_local_cif_matches.csv`
- `csd_identifiers_not_found_in_MOSAEC_local.csv`
- `csd_identifiers_not_found_in_MOSAEC_local.txt`

In [11]:
# Step 4 code - search MOSAEC full charged/neutral only and export one acquisition table
from pathlib import Path
import shutil
import pandas as pd

# Search only these two MOSAEC folders
full_charged_dir = Path(
    r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\Scraped DBs\MOSAEC-DB\mosaec-db_base\mosaec-db_base\full\charged"
)
full_neutral_dir = Path(
    r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\Scraped DBs\MOSAEC-DB\mosaec-db_base\mosaec-db_base\full\neutral"
)

cif_sources = [
    ("full_charged", full_charged_dir),
    ("full_neutral", full_neutral_dir),
]

missing_dirs = [f"{label}: {path}" for label, path in cif_sources if not path.exists()]
if missing_dirs:
    details = "\n".join(missing_dirs)
    raise FileNotFoundError(f"One or more requested folders were not found:\n{details}")


def normalize_code(value: str) -> str:
    return str(value).strip().upper()


def csd_match_with_digit_rule(csd_identifier: str, cif_stem: str) -> bool:
    """
    Match on first 6 characters, with extra rules:
    1) If CSD has no digit, CIF 7th character (if present) must not be a digit.
    2) If CSD 7th and 8th characters are digits, CIF 7th and 8th must match exactly.
    """
    csd = normalize_code(csd_identifier)
    stem = normalize_code(cif_stem)

    if len(csd) < 6 or len(stem) < 6:
        return False
    if stem[:6] != csd[:6]:
        return False

    csd_has_digit = any(ch.isdigit() for ch in csd)
    if (not csd_has_digit) and len(stem) >= 7 and stem[6].isdigit():
        return False

    csd_has_digit_suffix_7_8 = len(csd) >= 8 and csd[6].isdigit() and csd[7].isdigit()
    if csd_has_digit_suffix_7_8:
        if len(stem) < 8:
            return False
        if stem[6] != csd[6] or stem[7] != csd[7]:
            return False

    return True


# Use Step 3 output if present; otherwise reload from CSV
if "valid_csd_ids" not in globals() or not valid_csd_ids:
    csd_id_list_path = data_dir / "csd_identifiers_for_mosaec.csv"
    if not csd_id_list_path.exists():
        raise FileNotFoundError(
            "valid_csd_ids not found in memory and csd_identifiers_for_mosaec.csv is missing. "
            "Run Step 3 first."
        )
    valid_csd_ids = (
        pd.read_csv(csd_id_list_path)["csd_identifier"]
        .astype(str)
        .str.strip()
        .str.upper()
        .tolist()
    )

valid_csd_ids = sorted({normalize_code(c) for c in valid_csd_ids if str(c).strip()})

# Build candidate CIF table from the two allowed folders
candidate_rows = []
for source_label, folder in cif_sources:
    for cif_path in folder.rglob("*.cif"):
        candidate_rows.append(
            {
                "source_label": source_label,
                "filename": cif_path.name,
                "stem": cif_path.stem,
                "source_path": str(cif_path),
            }
        )

candidates_df = pd.DataFrame(candidate_rows)
if candidates_df.empty:
    raise RuntimeError("No CIF files were found in the requested MOSAEC folders.")

# For each requested CSD identifier, choose one CIF by source priority then filename
result_rows = []
for csd_id in valid_csd_ids:
    matches = candidates_df[candidates_df["stem"].apply(lambda s: csd_match_with_digit_rule(csd_id, s))].copy()

    if matches.empty:
        result_rows.append(
            {
                "csd_identifier": csd_id,
                "cif_source": "",
                "cif_filename": "",
                "database": "",
                "source_path": "",
            }
        )
        continue

    source_priority = {"full_charged": 0, "full_neutral": 1}
    matches["_priority"] = matches["source_label"].map(source_priority)
    best_match = matches.sort_values(["_priority", "filename"], ascending=[True, True]).iloc[0]

    result_rows.append(
        {
            "csd_identifier": csd_id,
            "cif_source": best_match["source_label"],
            "cif_filename": best_match["filename"],
            "database": "MOSAEC",
            "source_path": best_match["source_path"],
        }
    )

status_df = pd.DataFrame(result_rows)

# Copy all found CIFs into a dedicated output folder
found_cif_dir = data_dir / "MOSAEC_CIFs_found"
found_cif_dir.mkdir(parents=True, exist_ok=True)

for _, row in status_df[status_df["cif_filename"] != ""].iterrows():
    src = Path(row["source_path"])
    dst_name = f"{row['cif_source']}__{row['cif_filename']}"
    dst = found_cif_dir / dst_name
    shutil.copy2(src, dst)

# Keep only requested export columns
status_export_df = status_df[["csd_identifier", "cif_source", "cif_filename", "database"]].copy()

# Export exactly one file with one sheet
output_xlsx_path = data_dir / "mosaec_cif_acquisition_status.xlsx"
with pd.ExcelWriter(output_xlsx_path, engine="openpyxl") as writer:
    status_export_df.to_excel(writer, index=False, sheet_name="cif_acquisition")

print(f"Total target CSD identifiers: {len(valid_csd_ids)}")
print(f"Total CIF candidates scanned: {len(candidates_df)}")
print(f"CIF acquired: {(status_export_df['cif_filename'] != '').sum()}")
print(f"CIF not found: {(status_export_df['cif_filename'] == '').sum()}")
print(f"Found CIF output folder: {found_cif_dir}")
print(f"Saved: {output_xlsx_path}")

status_export_df.head(10)

KeyboardInterrupt: 

## Step 5 - Extend MOSAEC Results with Local CoRE-MOF Search

This step extends the MOSAEC acquisition table instead of creating a table only for unresolved IDs.

Workflow:
1. Load `mosaec_cif_acquisition_status.xlsx` from `data`.
2. Keep all rows (all CSD identifiers).
3. For rows where `cif_filename` is blank, search local CIF files under:
   - `C:/Users/james/OneDrive - Aix-Marseille Université/CNE Wroclaw 2026/CNE Thesis 2026/Scarped DBs/CoRE MOF - DB/All_solvent_removed_2019`
4. Apply the same identifier matching rules used in Step 4.
5. Save matched CoRE-MOF CIFs into `data/CoRE_MOF_CIFs_found/`.
6. Export one Excel file with one sheet and exactly four columns for all identifiers:
   - `csd_identifier`
   - `cif_source`
   - `cif_filename`
   - `database`

In [ ]:
# Step 5 code - extend MOSAEC acquisition table with local CoRE-MOF matches
from pathlib import Path
import shutil
import pandas as pd

project_dir = Path.cwd()
data_dir = project_dir / "data"

mosaec_status_path = data_dir / "mosaec_cif_acquisition_status.xlsx"
if not mosaec_status_path.exists():
    raise FileNotFoundError(
        f"MOSAEC status file not found: {mosaec_status_path}. Run Step 4 first."
    )

# Local CoRE-MOF dataset folder provided by user
core_mof_dir = Path(
    r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\Scraped DBs\CoRE MOF - DB\All_solvent_removed_2019"
)
if not core_mof_dir.exists():
    raise FileNotFoundError(f"Local CoRE-MOF folder not found: {core_mof_dir}")


def normalize_code(value: str) -> str:
    return str(value).strip().upper()


def name_prefix_before_separator(name: str) -> str:
    """Take only the initial alphanumeric block before separators like '_', '-', or space."""
    s = normalize_code(name)
    for sep in ("_", "-", " ", "."):
        if sep in s:
            s = s.split(sep, 1)[0]
    return s


def csd_match_with_digit_rule(csd_identifier: str, candidate_name: str) -> bool:
    """
    Match rules carried from Step 4:
    1) First 6 characters must match.
    2) If CSD has no digit, candidate 7th char (if present) must not be a digit.
    3) If CSD 7th and 8th chars are digits, candidate 7th/8th must match exactly.
    """
    csd = normalize_code(csd_identifier)
    cand = name_prefix_before_separator(candidate_name)

    if len(csd) < 6 or len(cand) < 6:
        return False
    if cand[:6] != csd[:6]:
        return False

    csd_has_digit = any(ch.isdigit() for ch in csd)
    if (not csd_has_digit) and len(cand) >= 7 and cand[6].isdigit():
        return False

    csd_has_digit_suffix_7_8 = len(csd) >= 8 and csd[6].isdigit() and csd[7].isdigit()
    if csd_has_digit_suffix_7_8:
        if len(cand) < 8:
            return False
        if cand[6] != csd[6] or cand[7] != csd[7]:
            return False

    return True


# Load MOSAEC status and keep all identifiers
mosaec_df = pd.read_excel(mosaec_status_path)
required_cols = {"csd_identifier", "cif_source", "cif_filename", "database"}
missing_cols = [c for c in required_cols if c not in mosaec_df.columns]
for col in missing_cols:
    mosaec_df[col] = ""

mosaec_df["csd_identifier"] = mosaec_df["csd_identifier"].astype(str).str.strip().str.upper()
mosaec_df["cif_source"] = mosaec_df["cif_source"].fillna("").astype(str).str.strip()
mosaec_df["cif_filename"] = mosaec_df["cif_filename"].fillna("").astype(str).str.strip()
mosaec_df["database"] = mosaec_df["database"].fillna("").astype(str).str.strip()

mosaec_df = mosaec_df[mosaec_df["csd_identifier"].str.lower() != "nan"].copy()

# Only unresolved IDs are searched in CoRE-MOF, but output keeps all rows
missing_ids = sorted(set(mosaec_df.loc[mosaec_df["cif_filename"] == "", "csd_identifier"].tolist()))

if missing_ids:
    # Build candidate list from local CoRE-MOF CIF files
    candidate_rows = []
    for cif_path in core_mof_dir.rglob("*.cif"):
        candidate_rows.append(
            {
                "entry_name": cif_path.stem,
                "source_path": str(cif_path),
                "filename": cif_path.name,
            }
        )

    core_candidates_df = pd.DataFrame(candidate_rows)
    if core_candidates_df.empty:
        raise RuntimeError(f"No CIF files found under local CoRE-MOF folder: {core_mof_dir}")

    found_core_dir = data_dir / "CoRE_MOF_CIFs_found"
    found_core_dir.mkdir(parents=True, exist_ok=True)

    core_found_map = {}
    for csd_id in missing_ids:
        matches = core_candidates_df[
            core_candidates_df["entry_name"].apply(lambda n: csd_match_with_digit_rule(csd_id, n))
        ].copy()

        if matches.empty:
            continue

        best = matches.sort_values(["filename"], ascending=[True]).iloc[0]

        src = Path(best["source_path"])
        out_name = f"ASR2019__{best['filename']}"
        out_path = found_core_dir / out_name
        shutil.copy2(src, out_path)

        core_found_map[csd_id] = {
            "cif_source": "asr2019_local",
            "cif_filename": out_name,
            "database": "CoRE-MOF",
        }
else:
    core_candidates_df = pd.DataFrame(columns=["entry_name", "source_path", "filename"])
    core_found_map = {}

# Extend MOSAEC table in place for unresolved rows that were found in CoRE-MOF
extended_df = mosaec_df[["csd_identifier", "cif_source", "cif_filename", "database"]].copy()

for idx, row in extended_df.iterrows():
    if row["cif_filename"] != "":
        # Keep existing MOSAEC hit unchanged
        continue

    csd_id = row["csd_identifier"]
    if csd_id in core_found_map:
        extended_df.at[idx, "cif_source"] = core_found_map[csd_id]["cif_source"]
        extended_df.at[idx, "cif_filename"] = core_found_map[csd_id]["cif_filename"]
        extended_df.at[idx, "database"] = core_found_map[csd_id]["database"]

# Export full extended table (all CSD identifiers)
core_output_path = data_dir / "mosaec_core_extended_cif_acquisition_status.xlsx"
with pd.ExcelWriter(core_output_path, engine="openpyxl") as writer:
    extended_df.to_excel(writer, index=False, sheet_name="cif_acquisition")

print(f"Total CSD identifiers in extended table: {len(extended_df)}")
print(f"Initially unresolved after MOSAEC: {len(missing_ids)}")
print(f"Resolved from local CoRE-MOF: {len(core_found_map)}")
print(f"Still unresolved: {(extended_df['cif_filename'] == '').sum()}")
print(f"Saved extended table: {core_output_path}")
if missing_ids:
    print(f"Local CoRE-MOF CIF candidates scanned: {len(core_candidates_df)}")
    print(f"Local CoRE-MOF found CIF folder: {data_dir / 'CoRE_MOF_CIFs_found'}")

extended_df.head(10)

Total CSD identifiers in extended table: 274
Initially unresolved after MOSAEC: 155
Resolved from local CoRE-MOF: 79
Still unresolved: 76
Saved extended table: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\data\mosaec_core_extended_cif_acquisition_status.xlsx
Local CoRE-MOF CIF candidates scanned: 12020
Local CoRE-MOF found CIF folder: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\data\CoRE_MOF_CIFs_found


,csd_identifier,cif_source,cif_filename,database
0,ACOCOM,asr2019_local,ASR2019__ACOCOM_clean.cif,CoRE-MOF
1,ADOGEH,full_neutral,ADOGEH_full.cif,MOSAEC
2,ADUROI,full_neutral,ADUROI_full.cif,MOSAEC
3,AJORAT,,,
4,ALAMUW,asr2019_local,ASR2019__ALAMUW_clean.cif,CoRE-MOF
5,AMAFOK,asr2019_local,ASR2019__AMAFOK_clean.cif,CoRE-MOF
6,AMUSOU,,,
7,AQIXEF,,,
8,ARAJEK,full_neutral,ARAJEK_full.cif,MOSAEC
9,AROFET,full_neutral,AROFET_full.cif,MOSAEC


## Step 6 - Resolve Remaining Missing CIFs from CCDC

This step is a final fallback after MOSAEC and local CoRE-MOF.

Workflow:
1. Load the extended table from Step 5 (`mosaec_core_extended_cif_acquisition_status.xlsx`).
2. Keep all rows, but query CCDC only for rows where `cif_filename` is still blank.
3. For each remaining `csd_identifier`, retrieve the entry directly from the CSD using the CCDC API.
4. Save found CIFs into `data/CCDC_CIFs_found/`.
5. Update unresolved rows in-place with CCDC acquisition metadata.
6. Export one final Excel file with one sheet and four columns containing all identifiers:
   - `csd_identifier`
   - `cif_source`
   - `cif_filename`
   - `database`

In [ ]:
# Step 6 code - fill remaining unresolved CSD identifiers from CCDC and export final combined table
from pathlib import Path
import pandas as pd

project_dir = Path.cwd()
data_dir = project_dir / "data"

extended_input_path = data_dir / "mosaec_core_extended_cif_acquisition_status.xlsx"
if not extended_input_path.exists():
    raise FileNotFoundError(
        f"Extended input file not found: {extended_input_path}. Run Step 5 first."
    )

# CCDC API imports (same dependency family used in notebooks 02/03)
try:
    import ccdc.io
except ImportError as exc:
    raise ImportError(
        "CCDC Python API is not available in this environment. Use the CCDC Python environment/kernel and rerun."
    ) from exc

# Load current combined table (MOSAEC + CoRE-MOF) and keep all rows
combined_df = pd.read_excel(extended_input_path)
for col in ["csd_identifier", "cif_source", "cif_filename", "database"]:
    if col not in combined_df.columns:
        combined_df[col] = ""

combined_df["csd_identifier"] = combined_df["csd_identifier"].astype(str).str.strip().str.upper()
combined_df["cif_source"] = combined_df["cif_source"].fillna("").astype(str).str.strip()
combined_df["cif_filename"] = combined_df["cif_filename"].fillna("").astype(str).str.strip()
combined_df["database"] = combined_df["database"].fillna("").astype(str).str.strip()
combined_df = combined_df[combined_df["csd_identifier"].str.lower() != "nan"].copy()

# Resolve only identifiers still missing a CIF
missing_mask = combined_df["cif_filename"] == ""
missing_ids = sorted(set(combined_df.loc[missing_mask, "csd_identifier"].tolist()))

ccdc_found_dir = data_dir / "CCDC_CIFs_found"
ccdc_found_dir.mkdir(parents=True, exist_ok=True)

reader = ccdc.io.EntryReader("CSD")
ccdc_found_map = {}
ccdc_not_found = []

for csd_id in missing_ids:
    try:
        entry = reader.entry(csd_id)
        if entry is None:
            ccdc_not_found.append(csd_id)
            continue

        crystal = entry.crystal
        if crystal is None:
            ccdc_not_found.append(csd_id)
            continue

        out_name = f"CCDC__{csd_id}.cif"
        out_path = ccdc_found_dir / out_name

        # Write CIF using CCDC writer
        with ccdc.io.CrystalWriter(str(out_path)) as writer:
            writer.write(crystal)

        ccdc_found_map[csd_id] = {
            "cif_source": "ccdc_api",
            "cif_filename": out_name,
            "database": "CCDC",
        }
    except Exception:
        ccdc_not_found.append(csd_id)

# Update unresolved rows in place with CCDC matches
for idx, row in combined_df.iterrows():
    if row["cif_filename"] != "":
        continue
    csd_id = row["csd_identifier"]
    if csd_id in ccdc_found_map:
        combined_df.at[idx, "cif_source"] = ccdc_found_map[csd_id]["cif_source"]
        combined_df.at[idx, "cif_filename"] = ccdc_found_map[csd_id]["cif_filename"]
        combined_df.at[idx, "database"] = ccdc_found_map[csd_id]["database"]

final_df = combined_df[["csd_identifier", "cif_source", "cif_filename", "database"]].copy()

# Final consolidated export: MOSAEC + CoRE-MOF + CCDC
final_output_path = data_dir / "mosaec_core_ccdc_final_cif_acquisition_status.xlsx"
with pd.ExcelWriter(final_output_path, engine="openpyxl") as writer:
    final_df.to_excel(writer, index=False, sheet_name="cif_acquisition")

print(f"Total CSD identifiers in final table: {len(final_df)}")
print(f"Missing before CCDC step: {len(missing_ids)}")
print(f"Resolved from CCDC: {len(ccdc_found_map)}")
print(f"Still unresolved after CCDC: {(final_df['cif_filename'] == '').sum()}")
print(f"CCDC CIF output folder: {ccdc_found_dir}")
print(f"Saved final combined table: {final_output_path}")

final_df.head(10)

Total CSD identifiers in final table: 274
Missing before CCDC step: 76
Resolved from CCDC: 75
Still unresolved after CCDC: 1
CCDC CIF output folder: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\data\CCDC_CIFs_found
Saved final combined table: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\data\mosaec_core_ccdc_final_cif_acquisition_status.xlsx


,csd_identifier,cif_source,cif_filename,database
0,ACOCOM,asr2019_local,ASR2019__ACOCOM_clean.cif,CoRE-MOF
1,ADOGEH,full_neutral,ADOGEH_full.cif,MOSAEC
2,ADUROI,full_neutral,ADUROI_full.cif,MOSAEC
3,AJORAT,ccdc_api,CCDC__AJORAT.cif,CCDC
4,ALAMUW,asr2019_local,ASR2019__ALAMUW_clean.cif,CoRE-MOF
5,AMAFOK,asr2019_local,ASR2019__AMAFOK_clean.cif,CoRE-MOF
6,AMUSOU,,,
7,AQIXEF,ccdc_api,CCDC__AQIXEF.cif,CCDC
8,ARAJEK,full_neutral,ARAJEK_full.cif,MOSAEC
9,AROFET,full_neutral,AROFET_full.cif,MOSAEC


## Step 7 - Fast clean-run copy of all isotherm JSON files

This step uses filename-only matching so rows with multiple DOIs in the sheet are still resolved correctly.

Workflow:
1. Load the master sheet (`MOF_CIF_Isotherms_MASTER_SHEET`) and read only `filename`.
2. Reuse `nistdb['Isotherms']` if already in memory; otherwise load `data/nistdb.pickle` (Notebook 01 style).
3. Resolve each source JSON from filename (DOI can be inferred from characters before `.iso` / `.Iso` when needed).
4. Enable clean-run safety: remove previous JSON files from the output folder before copying.
5. Copy all matched JSON files into `C:/Users/james/OneDrive - Aix-Marseille Université/CNE Wroclaw 2026/CNE Thesis 2026/MOF CIFs DATABASE - ALL/isotherms_full`.
6. Print summary only (no audit output file).

In [ ]:
# Step 7 code - fast clean-run copy using filename-only matching (no audit output file)
from pathlib import Path
from typing import Optional
import json
import pickle
import re
import shutil
import tempfile
import pandas as pd

project_dir = Path.cwd()
data_dir = project_dir / "data"

master_sheet_base = Path(
    r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF CIFs DATABASE - ALL\MOF_CIF_Isotherms_MASTER_SHEET"
 )
library_root = Path(
    r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\ML Project Directory\Data Collection\isodb-library\Library"
 )
output_dir = Path(
    r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF CIFs DATABASE - ALL\isotherms_full"
 )

nistdb_pickle_path = data_dir / "nistdb.pickle"
records_pickle_path = data_dir / "step7_isotherms_records.pickle"
filename_index_pickle_path = data_dir / "step7_filename_to_paths.pickle"

# Clean-run safety: delete existing JSON outputs before copying new ones.
CLEAN_RUN_OUTPUT_DIR = True

if not library_root.exists():
    raise FileNotFoundError(f"Library folder not found: {library_root}")

output_dir.mkdir(parents=True, exist_ok=True)


def resolve_master_sheet_path(base_path: Path) -> Path:
    """Resolve master sheet when extension is omitted."""
    if base_path.exists() and base_path.is_file():
        return base_path

    allowed_exts = (".xlsx", ".xls", ".xlsm", ".csv")
    direct_candidates = [
        base_path.with_suffix(ext)
        for ext in allowed_exts
        if base_path.with_suffix(ext).exists() and base_path.with_suffix(ext).is_file()
    ]

    if not direct_candidates:
        direct_candidates = sorted(
            p for p in base_path.parent.glob(f"{base_path.name}*")
            if p.is_file() and p.suffix.lower() in allowed_exts and not p.name.startswith("~$")
        )

    if not direct_candidates:
        raise FileNotFoundError(
            f"Could not find master sheet file for base path: {base_path}. "
            f"Checked Excel/CSV extensions and prefix matches."
        )

    if len(direct_candidates) > 1:
        print(f"Multiple master-sheet files found; using: {direct_candidates[0].name}")

    return direct_candidates[0]


def load_table(path: Path) -> pd.DataFrame:
    """Load CSV/Excel and fall back to a temporary copy if file is locked."""
    if path.suffix.lower() == ".csv":
        try:
            return pd.read_csv(path)
        except PermissionError:
            with tempfile.TemporaryDirectory() as tmpdir:
                tmp_path = Path(tmpdir) / path.name
                shutil.copy2(path, tmp_path)
                return pd.read_csv(tmp_path)

    try:
        return pd.read_excel(path)
    except PermissionError:
        with tempfile.TemporaryDirectory() as tmpdir:
            tmp_path = Path(tmpdir) / path.name
            shutil.copy2(path, tmp_path)
            return pd.read_excel(tmp_path)


def pick_column_name(df: pd.DataFrame, candidates: list[str], logical_name: str) -> str:
    """Pick the first matching column name case-insensitively."""
    lookup = {str(col).strip().lower(): col for col in df.columns}
    for candidate in candidates:
        col = lookup.get(candidate.lower())
        if col is not None:
            return col
    raise KeyError(
        f"Could not find '{logical_name}' column. Tried {candidates}. "
        f"Available columns: {list(df.columns)}"
    )


def normalize_doi(value: object) -> str:
    doi = str(value).strip()
    if not doi or doi.lower() in {"nan", "none"}:
        return ""
    doi = re.sub(r"^https?://(dx\.)?doi\.org/", "", doi, flags=re.IGNORECASE)
    doi = doi.replace("\\", "/").strip().strip("/")
    return doi


def normalize_filename(value: object) -> str:
    if pd.isna(value):
        return ""
    name = str(value).strip()
    if not name or name.lower() in {"nan", "none"}:
        return ""
    if name.lower().endswith(".json"):
        name = name[:-5]
    return name


def extract_doi_from_filename(filename: str) -> str:
    """Infer DOI from filename as everything before '.iso' (case-insensitive)."""
    parts = re.split(r"(?i)\.iso", filename, maxsplit=1)
    return normalize_doi(parts[0]) if parts else ""


def doi_to_folder_name(doi: str) -> str:
    """ISODB DOI folders are named using DOI with '/' removed."""
    return normalize_doi(doi).replace("/", "")


def resolve_case_insensitive_subdir(parent: Path, folder_name: str) -> Optional[Path]:
    if not parent.exists() or not parent.is_dir():
        return None
    matches = [
        child for child in parent.iterdir()
        if child.is_dir() and child.name.casefold() == folder_name.casefold()
    ]
    if not matches:
        return None
    return sorted(matches, key=lambda p: p.name.casefold())[0]


def load_isotherm_records() -> tuple[list[dict], str]:
    """Load isotherm metadata from memory/pickle, fallback to one-time JSON scan."""
    if "nistdb" in globals():
        maybe_nistdb = globals()["nistdb"]
        if isinstance(maybe_nistdb, dict) and isinstance(maybe_nistdb.get("Isotherms"), list):
            return maybe_nistdb["Isotherms"], "kernel memory (nistdb)"

    if nistdb_pickle_path.exists():
        try:
            with open(nistdb_pickle_path, "rb") as f:
                cached = pickle.load(f)
            if isinstance(cached, dict) and isinstance(cached.get("Isotherms"), list):
                return cached["Isotherms"], f"pickle cache ({nistdb_pickle_path.name})"
        except Exception:
            pass

    if records_pickle_path.exists():
        try:
            with open(records_pickle_path, "rb") as f:
                cached_records = pickle.load(f)
            if isinstance(cached_records, list):
                return cached_records, f"step7 cache ({records_pickle_path.name})"
        except Exception:
            pass

    isotherm_records = []
    for iso_path in library_root.glob("10*/*.json"):
        with open(iso_path, "r", encoding="utf-8") as f:
            isotherm_records.append(json.load(f))

    records_pickle_path.parent.mkdir(parents=True, exist_ok=True)
    with open(records_pickle_path, "wb") as f:
        pickle.dump(isotherm_records, f)

    return isotherm_records, f"library JSON scan ({len(isotherm_records)} records)"


def build_or_load_filename_index() -> tuple[dict[str, list[Path]], str]:
    """Build/load filename-stem index for fast fallback path resolution."""
    if filename_index_pickle_path.exists():
        try:
            with open(filename_index_pickle_path, "rb") as f:
                raw = pickle.load(f)
            index = {}
            if isinstance(raw, dict):
                for stem_key, str_paths in raw.items():
                    if not isinstance(stem_key, str) or not isinstance(str_paths, list):
                        continue
                    paths = [Path(p) for p in str_paths if isinstance(p, str)]
                    if paths:
                        index[stem_key] = paths
            if index:
                return index, f"filename index cache ({filename_index_pickle_path.name})"
        except Exception:
            pass

    index = {}
    for p in library_root.glob("10*/*.json"):
        key = p.stem.casefold()
        index.setdefault(key, []).append(p)

    filename_index_pickle_path.parent.mkdir(parents=True, exist_ok=True)
    with open(filename_index_pickle_path, "wb") as f:
        pickle.dump({k: [str(p) for p in v] for k, v in index.items()}, f)

    return index, f"library filename scan ({len(index)} unique stems)"


def resolve_source_json(
    filename: str,
    folder_cache: dict[str, Optional[Path]],
    folder_stem_map_cache: dict[str, dict[str, Path]],
    filename_index: dict[str, list[Path]],
) -> tuple[Optional[Path], str]:
    """Resolve source JSON path by filename using DOI-from-filename first, then global stem index."""
    inferred_doi = extract_doi_from_filename(filename)
    if inferred_doi:
        folder_name = doi_to_folder_name(inferred_doi)
        if folder_name not in folder_cache:
            direct = library_root / folder_name
            if direct.exists() and direct.is_dir():
                folder_cache[folder_name] = direct
            else:
                folder_cache[folder_name] = resolve_case_insensitive_subdir(library_root, folder_name)

        doi_folder = folder_cache[folder_name]
        if doi_folder is not None:
            direct_json = doi_folder / f"{filename}.json"
            if direct_json.exists() and direct_json.is_file():
                return direct_json, "inferred_doi_folder"

            if folder_name not in folder_stem_map_cache:
                folder_stem_map_cache[folder_name] = {
                    p.stem.casefold(): p
                    for p in doi_folder.glob("*.json")
                    if p.is_file()
                }

            hit = folder_stem_map_cache[folder_name].get(filename.casefold())
            if hit is not None and hit.exists() and hit.is_file():
                return hit, "inferred_doi_folder_ci"

    # Fallback by filename only across full library index
    paths = filename_index.get(filename.casefold(), [])
    if paths:
        return sorted(paths, key=lambda p: str(p).casefold())[0], "filename_index"

    return None, "not_found"


master_sheet_path = resolve_master_sheet_path(master_sheet_base)
master_df = load_table(master_sheet_path)
filename_col = pick_column_name(
    master_df,
    ["filename", "isotherm_filename", "file_name"],
    "filename",
)

isotherm_records, records_source = load_isotherm_records()
available_filenames = {
    normalize_filename(iso.get("filename", iso.get("isotherm_filename", iso.get("file_name", "")))).casefold()
    for iso in isotherm_records
}
available_filenames.discard("")

filename_index, filename_index_source = build_or_load_filename_index()

removed_existing = 0
if CLEAN_RUN_OUTPUT_DIR:
    for old_json in output_dir.glob("*.json"):
        old_json.unlink()
        removed_existing += 1

copied_count = 0
missing_filename = 0
missing_in_pickle_lookup = 0
missing_in_library = 0
copied_via_inferred_folder = 0
copied_via_filename_index = 0
row_collisions = 0
fallback_rows = []
collision_rows = []
folder_cache = {}
folder_stem_map_cache = {}

for row_idx, row in master_df.iterrows():
    filename = normalize_filename(row[filename_col])
    if not filename:
        missing_filename += 1
        continue

    if filename.casefold() not in available_filenames:
        missing_in_pickle_lookup += 1

    src_json, source_mode = resolve_source_json(
        filename,
        folder_cache=folder_cache,
        folder_stem_map_cache=folder_stem_map_cache,
        filename_index=filename_index,
    )

    if src_json is None:
        missing_in_library += 1
        continue

    if source_mode.startswith("inferred_doi_folder"):
        copied_via_inferred_folder += 1
    elif source_mode == "filename_index":
        copied_via_filename_index += 1
        fallback_rows.append(
            {
                "row": int(row_idx) + 2,
                "master_filename": filename,
                "source_json": src_json.name,
            }
        )

    dst_json = output_dir / src_json.name
    if dst_json.exists():
        row_collisions += 1
        collision_rows.append(
            {
                "row": int(row_idx) + 2,
                "master_filename": filename,
                "source_json": src_json.name,
            }
        )
        dst_json = output_dir / f"{src_json.stem}__row{int(row_idx) + 2}{src_json.suffix}"
        counter = 2
        while dst_json.exists():
            dst_json = output_dir / f"{src_json.stem}__row{int(row_idx) + 2}_{counter}{src_json.suffix}"
            counter += 1

    shutil.copyfile(src_json, dst_json)
    copied_count += 1

print(f"Master sheet used: {master_sheet_path}")
print(f"Isotherm metadata source: {records_source}")
print(f"Filename index source: {filename_index_source}")
print(f"Clean run enabled: {CLEAN_RUN_OUTPUT_DIR}")
print(f"Existing JSON removed before copy: {removed_existing}")
print(f"Rows scanned: {len(master_df)}")
print(f"JSON files copied: {copied_count}")
print(f"Rows skipped (missing filename): {missing_filename}")
print(f"Rows not seen in nistdb/pickle filename set: {missing_in_pickle_lookup}")
print(f"Rows with JSON not found in library: {missing_in_library}")
print(f"Copied via inferred DOI folder: {copied_via_inferred_folder}")
print(f"Copied via global filename index fallback: {copied_via_filename_index}")
print(f"Row-level destination name collisions handled: {row_collisions}")
print()
print("Fallback list (row | master filename | source json):")
if fallback_rows:
    for item in fallback_rows:
        print(f"- {item['row']} | {item['master_filename']} | {item['source_json']}")
else:
    print("- none")
print()
print("Collision list (row | master filename | source json):")
if collision_rows:
    for item in collision_rows:
        print(f"- {item['row']} | {item['master_filename']} | {item['source_json']}")
else:
    print("- none")
print(f"Output folder: {output_dir}")

Master sheet used: C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF CIFs DATABASE - ALL\MOF_CIF_Isotherms_MASTER_SHEET.xlsx
Isotherm metadata source: pickle cache (nistdb.pickle)
Filename index source: filename index cache (step7_filename_to_paths.pickle)
Clean run enabled: True
Existing JSON removed before copy: 399
Rows scanned: 399
JSON files copied: 399
Rows skipped (missing filename): 0
Rows not seen in nistdb/pickle filename set: 0
Rows with JSON not found in library: 0
Copied via inferred DOI folder: 393
Copied via global filename index fallback: 6
Row-level destination name collisions handled: 1

Fallback list (row | master filename | source json):
- 60 | 10.1007s10450-013-9564-x.isotherm4 | 10.1007s10450-013-9564-x.isotherm4.json
- 61 | 10.1007s10450-013-9564-x.isotherm5 | 10.1007s10450-013-9564-x.isotherm5.json
- 68 | 10.1007s10934-010-9378-0.Isotherm5 | 10.1007s10934-010-9378-0.Isotherm5.json
- 398 | 10.3969j.issn.1001-4861.2013.00.200.

## Step 8 - Unit audit and normalisation for exported isotherm JSON files

This step applies unit QA/QC to the JSON files copied in Step 7 (`isotherms_full`).

Workflow:
1. Load all JSON isotherms from the Step 7 output folder.
2. Apply filename-specific coherence fixes from notebook 01 (`Checking unique isotherms for units consistensy and coherence`):
   - pressure x*=100000/14.5038 for known under-scaled files
   - adsorption y*=1000/22414 for known mislabeled uptake files
   - pressure x*=1/100 for known over-scaled files
3. Run uptake unit audit/normalisation logic from notebook 01 (`Unit audit & normalisation`) to convert to `mmol/g`.
4. Run pressure unit audit/normalisation to convert recognised pressure units to `bar`.
5. Optionally skip files listed in `data/audit_removal_filenames.txt` (same list used in notebook 01).
6. Save all edits in place and export a full audit table to `data/step8_unit_normalisation_audit.csv`.

Notes:
- The coherence fixes include rerun guards (value-range checks) to reduce accidental double-application.
- Any file with unresolved/unknown units is listed in the printed summary and audit CSV.

In [14]:
# Step 8 code - unit audit/normalisation (mmol/g, bar) + coherence fixes from notebook 01
from pathlib import Path
import json
from collections import Counter
import pandas as pd

project_dir = Path.cwd()
data_dir = project_dir / "data"

# Use Step 7 output directory when available in kernel; fallback to the fixed project path.
isotherms_dir = output_dir if "output_dir" in globals() else Path(
    r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF CIFs DATABASE - ALL\isotherms_full"
)
if not isotherms_dir.exists():
    raise FileNotFoundError(f"Isotherm JSON folder not found: {isotherms_dir}")


def _norm_filename(name):
    s = str(name).strip()
    if s.lower().endswith(".json"):
        s = s[:-5]
    return s.lower()


def _load_filename_list(path: Path):
    out = set()
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.split("#", 1)[0].strip()
        if not line:
            continue
        parts = [p.strip() for p in line.split(",") if p.strip()]
        candidate = parts[-1] if parts else line
        out.add(_norm_filename(candidate))
    return out


def _max_pressure(iso):
    vals = [
        pt.get("pressure")
        for pt in iso.get("isotherm_data", [])
        if pt.get("pressure") is not None
    ]
    return max(vals) if vals else None


def _max_adsorption(iso):
    vals = [
        pt.get("total_adsorption")
        for pt in iso.get("isotherm_data", [])
        if pt.get("total_adsorption") is not None
    ]
    return max(vals) if vals else None


def _scale_pressure(iso, factor):
    for pt in iso.get("isotherm_data", []):
        p = pt.get("pressure")
        if p is not None:
            pt["pressure"] = p * factor


def _scale_adsorption(iso, factor):
    for pt in iso.get("isotherm_data", []):
        q = pt.get("total_adsorption")
        if q is not None:
            pt["total_adsorption"] = q * factor
        for sp in pt.get("species_data", []):
            aq = sp.get("adsorption")
            if aq is not None:
                sp["adsorption"] = aq * factor


M_CH4 = 16.043      # g/mol (methane, same as notebook 01 unit-normalisation block)
V_STP = 22_414.0    # cm3/mol at STP


def _adsorption_conversion_factor(unit):
    """Return multiplier for conversion to mmol/g; None when unit is unknown."""
    if unit is None:
        return None

    u = str(unit).strip().lower()
    u_nospace = u.replace(" ", "")

    if u == "mmol/g":
        return 1.0
    if u in ("mol/kg",):
        return 1.0
    if u in ("mmol/kg",):
        return 1e-3
    if u in ("µmol/g", "umol/g", "μmol /g") or u_nospace in ("µmol/g", "umol/g", "μmol/g"):
        return 1e-3
    if u in ("mg/g",):
        return 1.0 / M_CH4
    if u in ("g/g",):
        return 1000.0 / M_CH4
    if u in ("g/l",):
        return 1.0 / M_CH4
    if u in ("g/ml",):
        return 1000.0 / M_CH4
    if u in (
        "cm3(stp)/g", "cc(stp)/g", "ml(stp)/g",
        "cm3 (stp)/g", "cc (stp)/g", "ml (stp)/g", "ml/g",
    ):
        return 1000.0 / V_STP
    if u in ("l(stp)/g", "l (stp)/g"):
        return 1000000.0 / V_STP
    if u in ("wt%", "wt. %", "wt.%", "weight %", "uptake%", "uptake %"):
        return 10.0 / M_CH4

    # Extended aliases from isodb-library conversion utility.
    if u in ("mol/g",):
        return 1000.0
    if u in ("cm3/g", "cc/g"):
        return 1000.0 / V_STP
    if u in ("l/g",):
        return 1000000.0 / V_STP
    if u in ("mg/kg",):
        return 1e-3 / M_CH4

    return None


def _pressure_to_bar_factor(unit):
    """Return multiplier for conversion to bar; None when unit is unknown."""
    if unit is None:
        return None

    u = str(unit).strip().lower().replace(" ", "")
    if u in ("", "nan", "none"):
        return None

    mapping = {
        "bar": 1.0,
        "bars": 1.0,
        "pa": 1.0 / 100000.0,
        "kpa": 1.0 / 100.0,
        "mpa": 10.0,
        "mbar": 1.0 / 1000.0,
        "atm": 1.01325,
        "atmosphere": 1.01325,
        "psi": 0.0689475729,
        "torr": 0.001333223684,
        "mmhg": 0.001333223684,
    }
    return mapping.get(u)


if "filtered" not in globals():
    print("Warning: 'filtered' not found in kernel; Step 8 runs directly on JSON files.")

# Coherence-fix filename sets copied from notebook 01 ('Checking unique isotherms...').
psi_to_bar_filenames = {
    "10.1021jp304631m.Isotherm15",
    "10.1021jp304631m.Isotherm17",
    "10.1021jp304631m.Isotherm30",
    "10.1021jp304631m.Isotherm31",
    "10.1021jp304631m.Isotherm39",
    "10.1021jp304631m.Isotherm40",
}
cm3_to_mmol_filenames = {
    "10.1039C3ta11548h.isotherm17",
    "10.1039C3ta11548h.isotherm18",
    "10.1039C3ta11548h.isotherm15",
    "10.1039C3ta11548h.isotherm16",
    "10.1039C3ta11840a.Isotherm6",
}
divide_by_100_filenames = {
    "10.1039C3cc48275h.Isotherm10",
    "10.1039C3cc48275h.Isotherm11",
    "10.1039C3cc48275h.Isotherm12",
    "10.1039C3cc48275h.Isotherm13",
    "10.1039C3cc48275h.Isotherm14",
    "10.1016j.micromeso.2011.09.006.isotherm5",
}

psi_to_bar_norm = {_norm_filename(x) for x in psi_to_bar_filenames}
cm3_to_mmol_norm = {_norm_filename(x) for x in cm3_to_mmol_filenames}
divide_by_100_norm = {_norm_filename(x) for x in divide_by_100_filenames}

pressure_factor_psi = 100000.0 / 14.5038
adsorption_factor_cm3 = 1000.0 / 22414.0
pressure_factor_div100 = 1.0 / 100.0

# Keep this aligned with notebook 01 step 1 behavior (skip listed files from processing).
APPLY_AUDIT_REMOVAL_SKIP = True
audit_removal_file = data_dir / "audit_removal_filenames.txt"
audit_remove_norm = _load_filename_list(audit_removal_file) if audit_removal_file.exists() else set()

json_paths = sorted(isotherms_dir.glob("*.json"))
if not json_paths:
    raise RuntimeError(f"No JSON files found in {isotherms_dir}")

records = []
updated_files = 0
skipped_by_audit = 0

coherence_step2_count = 0
coherence_step3_count = 0
coherence_step4_count = 0
adsorption_label_converted_count = 0
pressure_label_converted_count = 0
unresolved_adsorption_count = 0
unresolved_pressure_count = 0

for path in json_paths:
    with open(path, "r", encoding="utf-8") as f:
        iso = json.load(f)

    filename = iso.get("filename", path.stem)
    fn_norm = _norm_filename(filename)

    ads_before = iso.get("adsorptionUnits", None)
    pressure_before = iso.get("pressureUnits", None)
    coherence_fixes = []
    modified = False

    if APPLY_AUDIT_REMOVAL_SKIP and fn_norm in audit_remove_norm:
        skipped_by_audit += 1
        records.append(
            {
                "filename": filename,
                "source_json": path.name,
                "status": "skipped_audit_removal",
                "adsorptionUnits_before": ads_before,
                "adsorptionUnits_after": ads_before,
                "pressureUnits_before": pressure_before,
                "pressureUnits_after": pressure_before,
                "coherence_fixes": "",
                "adsorption_factor_label": "",
                "pressure_factor_label": "",
                "unresolved_adsorption_unit": "",
                "unresolved_pressure_unit": "",
            }
        )
        continue

    # Coherence fix STEP 2 from notebook 01 (guarded to avoid double-application).
    max_p = _max_pressure(iso)
    if fn_norm in psi_to_bar_norm and max_p is not None and max_p < 0.02:
        _scale_pressure(iso, pressure_factor_psi)
        coherence_step2_count += 1
        coherence_fixes.append("x*=100000/14.5038")
        modified = True

    # Coherence fix STEP 3 from notebook 01 (guarded to avoid double-application).
    max_q = _max_adsorption(iso)
    if fn_norm in cm3_to_mmol_norm and max_q is not None and max_q > 2.0:
        _scale_adsorption(iso, adsorption_factor_cm3)
        coherence_step3_count += 1
        coherence_fixes.append("y*=1000/22414")
        modified = True

    # Coherence fix STEP 4 from notebook 01 (guarded to avoid double-application).
    max_p = _max_pressure(iso)
    if fn_norm in divide_by_100_norm and max_p is not None and max_p > 100.0:
        _scale_pressure(iso, pressure_factor_div100)
        coherence_step4_count += 1
        coherence_fixes.append("x*=1/100")
        modified = True

    unresolved_adsorption_unit = ""
    unresolved_pressure_unit = ""

    # Uptake-unit normalisation logic from notebook 01.
    ads_factor = _adsorption_conversion_factor(iso.get("adsorptionUnits", None))
    if ads_factor is None:
        unresolved_adsorption_count += 1
        unresolved_adsorption_unit = str(iso.get("adsorptionUnits", ""))
    else:
        if ads_factor != 1.0:
            _scale_adsorption(iso, ads_factor)
            adsorption_label_converted_count += 1
            modified = True
        if iso.get("adsorptionUnits") != "mmol/g":
            iso["adsorptionUnits"] = "mmol/g"
            modified = True

    # Pressure-unit normalisation to bar.
    pressure_factor = _pressure_to_bar_factor(iso.get("pressureUnits", None))
    if pressure_factor is None:
        unresolved_pressure_count += 1
        unresolved_pressure_unit = str(iso.get("pressureUnits", ""))
    else:
        if pressure_factor != 1.0:
            _scale_pressure(iso, pressure_factor)
            pressure_label_converted_count += 1
            modified = True
        if iso.get("pressureUnits") != "bar":
            iso["pressureUnits"] = "bar"
            modified = True

    ads_after = iso.get("adsorptionUnits", None)
    pressure_after = iso.get("pressureUnits", None)

    if modified:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(iso, f, indent=4)
            f.write("\n")
        updated_files += 1

    records.append(
        {
            "filename": filename,
            "source_json": path.name,
            "status": "updated" if modified else "checked_no_change",
            "adsorptionUnits_before": ads_before,
            "adsorptionUnits_after": ads_after,
            "pressureUnits_before": pressure_before,
            "pressureUnits_after": pressure_after,
            "coherence_fixes": "; ".join(coherence_fixes),
            "adsorption_factor_label": ads_factor if ads_factor is not None else "",
            "pressure_factor_label": pressure_factor if pressure_factor is not None else "",
            "unresolved_adsorption_unit": unresolved_adsorption_unit,
            "unresolved_pressure_unit": unresolved_pressure_unit,
        }
    )

step8_audit_df = pd.DataFrame(records)

audit_csv = data_dir / "step8_unit_normalisation_audit.csv"
step8_audit_df.to_csv(audit_csv, index=False)

# Compact unit summaries excluding files skipped by audit-removal list.
active_df = step8_audit_df[step8_audit_df["status"] != "skipped_audit_removal"].copy()
ads_before_counts = Counter(active_df["adsorptionUnits_before"].astype(str))
ads_after_counts = Counter(active_df["adsorptionUnits_after"].astype(str))
pressure_before_counts = Counter(active_df["pressureUnits_before"].astype(str))
pressure_after_counts = Counter(active_df["pressureUnits_after"].astype(str))

not_mmol = active_df[active_df["adsorptionUnits_after"].astype(str) != "mmol/g"]
not_bar = active_df[active_df["pressureUnits_after"].astype(str) != "bar"]

print(f"JSON files scanned: {len(json_paths)}")
print(f"Skipped by audit-removal list: {skipped_by_audit}")
print(f"Files updated in place: {updated_files}")
print()
print("Coherence fixes applied (from notebook 01 unique-isotherm section):")
print(f"- Step 2 pressure x*=100000/14.5038: {coherence_step2_count}")
print(f"- Step 3 adsorption y*=1000/22414: {coherence_step3_count}")
print(f"- Step 4 pressure x*=1/100: {coherence_step4_count}")
print()
print("Label-based unit normalisation:")
print(f"- Adsorption converted by unit label: {adsorption_label_converted_count}")
print(f"- Pressure converted by unit label: {pressure_label_converted_count}")
print(f"- Unresolved adsorption units: {unresolved_adsorption_count}")
print(f"- Unresolved pressure units: {unresolved_pressure_count}")
print()
print("Adsorption units BEFORE:", dict(sorted(ads_before_counts.items(), key=lambda kv: kv[0])))
print("Adsorption units AFTER :", dict(sorted(ads_after_counts.items(), key=lambda kv: kv[0])))
print("Pressure units BEFORE  :", dict(sorted(pressure_before_counts.items(), key=lambda kv: kv[0])))
print("Pressure units AFTER   :", dict(sorted(pressure_after_counts.items(), key=lambda kv: kv[0])))
print()
print(f"Files not in mmol/g after Step 8: {len(not_mmol)}")
if len(not_mmol):
    print(not_mmol[["filename", "adsorptionUnits_after", "unresolved_adsorption_unit"]].head(20).to_string(index=False))
print(f"Files not in bar after Step 8: {len(not_bar)}")
if len(not_bar):
    print(not_bar[["filename", "pressureUnits_after", "unresolved_pressure_unit"]].head(20).to_string(index=False))
print()
print(f"Saved audit CSV: {audit_csv}")

step8_not_mmol = not_mmol
step8_not_bar = not_bar

JSON files scanned: 399
Skipped by audit-removal list: 0
Files updated in place: 0

Coherence fixes applied (from notebook 01 unique-isotherm section):
- Step 2 pressure x*=100000/14.5038: 0
- Step 3 adsorption y*=1000/22414: 0
- Step 4 pressure x*=1/100: 0

Label-based unit normalisation:
- Adsorption converted by unit label: 0
- Pressure converted by unit label: 0
- Unresolved adsorption units: 0
- Unresolved pressure units: 0

Adsorption units BEFORE: {'mmol/g': 399}
Adsorption units AFTER : {'mmol/g': 399}
Pressure units BEFORE  : {'bar': 399}
Pressure units AFTER   : {'bar': 399}

Files not in mmol/g after Step 8: 0
Files not in bar after Step 8: 0

Saved audit CSV: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\ML Project Directory\Data Collection\data\step8_unit_normalisation_audit.csv


In [15]:
# Step 8b - targeted pressure coherence fix for 10.1016j.micromeso.2011.09.006.isotherm5
from pathlib import Path
import json
import pandas as pd

project_dir = Path.cwd()
data_dir = project_dir / "data"
target_filename = "10.1016j.micromeso.2011.09.006.isotherm5"
target_json_name = f"{target_filename}.json"

isotherms_dir = output_dir if "output_dir" in globals() else Path(
    r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF CIFs DATABASE - ALL\isotherms_full"
)
target_path = isotherms_dir / target_json_name
if not target_path.exists():
    raise FileNotFoundError(f"Target isotherm JSON not found: {target_path}")

with open(target_path, "r", encoding="utf-8") as f:
    iso = json.load(f)

pressures = [pt.get("pressure") for pt in iso.get("isotherm_data", []) if pt.get("pressure") is not None]
if not pressures:
    raise RuntimeError(f"No pressure data found in {target_json_name}")

max_p_before = max(pressures)
applied = False

# Idempotent guard: the known incorrect case sits around ~97 and should become ~0.97 after /100.
if max_p_before > 40.0:
    for pt in iso.get("isotherm_data", []):
        p = pt.get("pressure")
        if p is not None:
            pt["pressure"] = p / 100.0
    iso["pressureUnits"] = "bar"
    applied = True

with open(target_path, "w", encoding="utf-8") as f:
    json.dump(iso, f, indent=4)
    f.write("\n")

pressures_after = [pt.get("pressure") for pt in iso.get("isotherm_data", []) if pt.get("pressure") is not None]
max_p_after = max(pressures_after) if pressures_after else None

audit_csv = data_dir / "step8_unit_normalisation_audit.csv"
if audit_csv.exists():
    audit_df = pd.read_csv(audit_csv)
    m = audit_df["filename"].astype(str).str.lower() == target_filename.lower()
    if m.any():
        idx = audit_df.index[m][0]
        prev_fix = str(audit_df.at[idx, "coherence_fixes"]) if "coherence_fixes" in audit_df.columns else ""
        if prev_fix in {"nan", "None"}:
            prev_fix = ""
        if applied and "x*=1/100" not in prev_fix:
            audit_df.at[idx, "coherence_fixes"] = (prev_fix + "; x*=1/100").strip("; ").strip()
        audit_df.at[idx, "status"] = "updated_target_pressure_fix" if applied else str(audit_df.at[idx, "status"])
        if "pressureUnits_after" in audit_df.columns:
            audit_df.at[idx, "pressureUnits_after"] = "bar"
        if "pressure_factor_label" in audit_df.columns:
            audit_df.at[idx, "pressure_factor_label"] = 1.0
        audit_df.to_csv(audit_csv, index=False)

print(f"Target file: {target_json_name}")
print(f"Max pressure before: {max_p_before}")
print(f"Applied /100 correction: {applied}")
print(f"Max pressure after: {max_p_after}")
print(f"Updated JSON: {target_path}")
print(f"Updated audit CSV: {audit_csv}")

Target file: 10.1016j.micromeso.2011.09.006.isotherm5.json
Max pressure before: 97.0033
Applied /100 correction: True
Max pressure after: 0.9700329999999999
Updated JSON: C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF CIFs DATABASE - ALL\isotherms_full\10.1016j.micromeso.2011.09.006.isotherm5.json
Updated audit CSV: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\ML Project Directory\Data Collection\data\step8_unit_normalisation_audit.csv


C:\Users\james\AppData\Local\Temp\ipykernel_20380\1109443481.py:54: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'x*=1/100' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  audit_df.at[idx, "coherence_fixes"] = (prev_fix + "; x*=1/100").strip("; ").strip()
